# <span style="color:DodgerBlue">**I. Getting a Time-series of Vegetation Indices**</span>
---

The tutorial focuses on a forest plot that has been exposed to a bark beetle attack. It aims to map the spatio-temporal evolution of this phenomenom. Thus we will analyse a satellite time-series and derived vegetation indices.
Before analyzing the time-series, we propose here to search through a STAC (*Spatio-temporal Assets Catalog*) catalog the available images according to several criteria (start date, end date, cloudiness, etc.), then calculate vegetation indices and store them.

---
## <span style="color:DodgerBlue">0. Install packages</span>

Depending of your Python environment, you may have to install the following packages in your current session (comment the line of code if necessary).

In [ ]:
!pip install pystac-client
!pip install planetary_computer
!pip install xarray 
!pip install rioxarray
!pip install odc-stac
!pip install geopandas
!pip install netCDF4

## <span style="color:DodgerBlue">1. Prerequisites</span>
### <span style="color:DodgerBlue">1.1. Import librairies</span>

The librairies are essentially tools for:
- requesting data in STAC catalog(s);
- manipulating and processing geospatial data.

In [ ]:
import os

# data storage via Google Drive
from google.colab import drive

# STAC API
from pystac_client import Client
import planetary_computer

# xarray
import xarray as xr
import rioxarray as rio
import numpy as np

# Geospatial librairies
import geopandas as gpd
import odc
from odc.stac import stac_load
from odc.geo.geobox import GeoBox
from rasterio.crs import CRS

# ignore warnings messages
import warnings
warnings.filterwarnings('ignore') 

# to save some variables
import pickle

### <span style="color:DodgerBlue">1.2. Mounting Google Drive</span>

For practical reasons, all the data produced via the different notebooks will be stored in your **Google Drive** space, in a repository named `nrt_data`. 

In [ ]:
# mounting
mount_dir = '/content/drive'
drive.mount(mount_dir)

# create a new rep in Google Drive account
work_dir = os.path.join(mount_dir, 'My Drive/nrt_data')
if not os.path.exists(work_dir):
    os.mkdir(work_dir)

---
## <span style="color:DodgerBlue">2. Request in a STAC catalog</span>

For this example, we search satellite images into the **Microsoft Planetary Computer** (mpc). It is possible to use other [catalogs](https://stacspec.org/en/about/datasets/), you must know the related URL and the desired collection.

In [ ]:
s2_stac = {'mpc':{'stac':"https://planetarycomputer.microsoft.com/api/stac/v1",
                  'coll':"sentinel-2-l2a",
                  'key_sat':"s2",
                  'modifier':planetary_computer.sign_inplace}
          }

### <span style="color:DodgerBlue">2.1 Search parameters</span>

Beyond the **satellite collection** you want, it is necessary to define the **location**. Here, we use the bounding box of the vector layer *bd-foret_p33.geojson*, stored in the GitHub NRT-tutorial repository.


In [ ]:
# downloading vector file from git to colab env
!mkdir -p nrt_data
![ ! -f nrt_data/bd-foret_p33.geojson ] && wget https://raw.githubusercontent.com/kenoz/NRT-tutorial/colab_version/data_ref/bd-foret_p33.geojson -P nrt_data

For location filter, the bounding box coordinates must be given in Lat/Long i.e. the code *"EPSG:4326"* (WGS 84).

In [ ]:
# loading vector
vector = gpd.read_file(r"nrt_data/bd-foret_p33.geojson")

# bbox in LAT/LON
vector_4326 = vector.to_crs("EPSG:4326")
aoi_bounds = vector_4326.bounds.values.tolist()[0]

# bbox in EPSG 3035
vector_3035 = vector.to_crs("EPSG:3035")
aoi_bounds_3035 = vector_3035.bounds.values.tolist()[0]

The other filters describe the **time range** and the maximum **cloud percentage**. Here, the cloud percentage is fixed at 100% so that we get all the available images.

In [ ]:
# output directory
output_dir = os.path.join(work_dir, 'output')
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

# time range
startdate = "2017-01"
enddate = "2020-12"
time_range = "{}/{}".format(startdate, enddate)

# cloud percentage
cloud_nb = 100

Here, we keep some basic variables that will be useful for finding produced files throughout the tutorials.

In [ ]:
# saving variables
dict_var = {}
dict_var['output_dir'] = output_dir
dict_var['startdate'] = startdate
dict_var['enddate'] = enddate

with open(os.path.join(output_dir, 'nrt_var.txt'), 'wb') as file:
    pickle.dump(dict_var, file)


### <span style="color:DodgerBlue">2.2. pySTAC-Client: list of collection items</span>

Now the search parameters being defined, we can search images in the MPC STAC catalog.

In [ ]:
provider = 'mpc'
stac = s2_stac[provider]

catalog = Client.open(stac['stac'],
                      modifier=stac['modifier'])

search = catalog.search(
                    collections=[stac['coll']],
                    bbox=aoi_bounds,
                    datetime=time_range,
                    method='GET',
                    query={"eo:cloud_cover": {"lt": cloud_nb}},
                    sortby="datetime"
                    )

# convert stac catalog into item collection
items = search.item_collection()

We can count the number of images that correspond to the search parameters.

In [ ]:
print(f"{len(items)} items found")

Then, we explore the metadata of the first image.

In [ ]:
items[0]

---
## <span style="color:DodgerBlue">3. Creation of Datacube</span>

All Sentinel-2 bands are kept (60m bands excepted). The Datacube dimensions are defined by the bounding box in CRS *EPSG:3035*.

### <span style="color:DodgerBlue">3.1. Geobox definition</span>

First, we define the GeoBox object. This variable contains:
- the coordinates of the bounding box,
- the coordinate reference system (i.e. projection system),
- the output spatial resolution of pixels.

In [ ]:
crs = CRS.from_epsg(3035)
geobox = GeoBox.from_bbox(odc.geo.geom.BoundingBox(*aoi_bounds_3035),
                          crs=crs,
                          resolution=10)

geobox

### <span style="color:DodgerBlue">3.2. Loading the Data Cube (xarray)</span>

We load the datacube object, here a xarray DataSet, thanks to **odc-stac** package.

In [ ]:
bands = ['B02', 'B03', 'B04', 'B05', 'B06',
         'B07', 'B08', 'B8A', 'B11', 'B12', 
         'SCL']

array = stac_load(items,
                  bands=bands,
                  groupby='solar_day',
                  chunks={'x': 640, 'y': 640},
                  geobox=geobox,
                  fail_on_error=False)

array

- An `xarray.Dataset` is a collection of multiple variables (`xarray.DataArray`) with shared dimensions (x, y, time). 
- An `xarray.DataArray` is a single multi-dimensional array. 

Here, `array` is an `xarray.Dataset` with Sentinel-2 bands as variables. We can access to a `xarray.DataArray` with the following instruction (here, we want the band 'B02'):

In [ ]:
array.B02

---
### <span style="color:DodgerBlue">3.3. Writing array into file(s)</span> <span style="color:red">[OPTIONNAL AND NOT RECOMMENDED HERE]</span>

We can store the Datacube in a file. There are severale strategies depending of your needs, for example:
- saving one multispectral image per date,
- saving the whole datacube in only one file (then you have to choose an appropriate format).

**Be careful**, the current time-series is quite large in terms of dates and bands (523 dates x 11 bands). The following commented code presents how to store the datacube. Depending of your computing ressources, it can be **quite long** (about 13 min in CDSE-hub, 20 min in Colab).

#### <span style="color:DodgerBlue">3.3.1. Option 1: one Geotiff file per date</span>

*N.B.: Uncomment the following code to run it and produce the output files.*

In [ ]:
"""
print(len(array.time))
for i in range(len(array.time)):
    test = array.isel(time=i)
    date = test.coords["time"]
    datetxt = np.datetime_as_string(date, unit='D')
    #print('write ', datetxt)
    test.rio.to_raster(os.path.join(output_dir, f"S2_plot01_{datetxt}.tif"), compress='LZW')
"""

#### <span style="color:DodgerBlue">3.3.2. Option 2: a NetCDF file for the whole time-series</span>

*N.B.: Uncomment the following code to run it and produce the output NetCDF file.*


In [ ]:
"""
array.to_netcdf(
    os.path.join(output_dir, f'S2TS_{startdate}-{enddate}.nc')
    )
"""

To open a NetCDF File as Array, you can use the following code:

In [ ]:
"""
ds_disk = xr.open_dataset(
    os.path.join(output_dir, f'S2TS_{startdate}-{enddate}.nc')
    )
ds_disk
"""

---

## <span style="color:DodgerBlue">4. Data reduction into univariate series</span>

### <span style="color:DodgerBlue">4.1. Computing vegetation indices (VI)</span>

We calculate three vegetation indices:
- **NDVI** (*Normalized Difference Vegetation Index*): index used to quantify vegetation greenness and is useful in understanding vegetation density and assessing changes in plant health (*source: USGS*).
- **NDMI** (*Normalized Difference Moisture Index*): index used to determine vegetation water content. It is calculated as a ratio between the NIR and SWIR values in traditional fashion (*source: USGS*).
- **CRSWIR** (*Continuum Removal SWIR*): index sensitive to the vegetation water content, and calculated from the near and mid-infrared bands. this involves highlighting the absorption phenomena associated with vegetation water, measurable at the level of band 11 of Sentinel-2, centered in the short-wave infrared (SWiR) around 1600 nm. This index is based on the Continuum Removal (CR) technique which consists of maximizing the spectral contrast associated with absorption photos, by normalizing the reflectance value in relation to the value of a 'convex envelope' calculated from of neighboring spectral bands (*source: Dutrieux R., Feret J.B., Ose K. 2021. Mise au point d'une methode reproductible pour le suivi generalise des degats de scolytes par teledetection satellitaire, ONF-RDVT 69-70,pp.37-44* -> [web link](https://www.onf.fr/onf/+/cec::les-rendez-vous-techniques-de-lonf-no69-70.html)).

#### <span style="color:DodgerBlue">4.1.1. Inserting VI variables in Datacube</span>

We create a new Datacube variable for each vegetation index. Then, the spectral bands are dropped. Though it is not mandatory, it allows to save storage/memory space.

In [ ]:
array['ndvi'] = (array.B8A - array.B04) / (array.B8A + array.B04)
array['ndmi'] = (array.B8A - array.B11) / (array.B8A + array.B11)
array['crswir'] = array.B11/(array.B8A+((array.B12-array.B8A)/(2185.7-864))*(1610.4-864))

# add 'grid_mapping' attribute to keep CRS coordinates (cf. export into netCDF)
array.ndvi.attrs = {'grid_mapping': 'spatial_ref'}
array.ndmi.attrs = {'grid_mapping': 'spatial_ref'}
array.crswir.attrs = {'grid_mapping': 'spatial_ref'}

# remove spectral bands
array = array.drop_vars(['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12'])

#### <span style="color:DodgerBlue">4.1.2. Saving the VI Datacube</span> <span style="color:red">[OPTIONNAL AND NOT RECOMMENDED HERE]</span>

**Be careful**, even though the current time-series is only made of three bands (VI indices) + the SLC band, the following commented code can be **quite long** (about 12 min in Colab).

In [ ]:
"""
array.to_netcdf(
    os.path.join(output_dir, f'S2TS_{startdate}-{enddate}_vi.nc')
    )
"""

### <span style="color:DodgerBlue">4.2. Cloud masks and filtering</span>

#### <span style="color:DodgerBlue">4.2.1. Applying cloud masks</span>

The Level-2A processing includes a Scene Classification and an Atmospheric Correction applied to Top-Of-Atmosphere (TOA) Level-1C orthoimage products. The Scene Classification (SCL) was developed to distinguish between cloudy pixels, clear pixels and water pixels of Sentinel-2 data and is a result of the Scene classification algorithm run by ESA. 

![fig_nrt](img/SCL.png)

We apply a cloud mask made from SCL (cloud shadows, clouds and cirrus) in order not to consider cloudy pixels in the next analyses.

In [ ]:
# mask creation
cloud_mask = array.SCL.isin([3, 8, 9, 10])
array_mask = array.where(~cloud_mask)

print(f"number of dates {len(array_mask.time)}")

#### <span style="color:DodgerBlue">4.2.2. Keeping images without clouds</span>
We can also decide to remove all images that are partially or totally covered by clouds and shadows.

In [ ]:
test_mask = cloud_mask.sum(dim=['x', 'y']) == 0
array_subset = array_mask.where(test_mask.compute(), drop=True)

print(f"number of dates {len(array_subset.time)}")

#### <span style="color:DodgerBlue">4.2.3. Keeping images according to a clouds threshold</span>

The previous request is a little too strong. We finally prefer to filter the images according to a percentage of cloudy pixels.

To calculate it, we need to know the dimension of our data cube in x and y.

In [ ]:
size = zip(cloud_mask.dims, cloud_mask.shape)
size = list(size)
size

We deduce the total number of pixel for each layer of the datacube.

In [ ]:
y = [i[1] for i in size if 'y' in i][0]
x = [i[1] for i in size if 'x' in i][0]

pxl_tot = x*y
pxl_tot

Now, it is up to the user to define the most suitable clouds threshold in the `cld_max` variable (here 0.5, i.e. 50%).

In [ ]:
cld_max = 0.5
test_mask = cloud_mask.sum(dim=['x', 'y'])/pxl_tot <= cld_max
array_cldpct = array_mask.where(test_mask.compute(), drop=True)

print(f"number of dates {len(array_cldpct.time)}")

#### <span style="color:DodgerBlue">4.2.4. Saving the filtered Datacube of vegetation indices</span>

For the next tutorials, we will only use:
- the VI Datacube without any clouds.

In [ ]:
array_subset.to_netcdf(
    os.path.join(output_dir, f'S2TS_{startdate}-{enddate}_vi-nocloud.nc')
    )

- the Datacube wiht a clouds percentage threshold less than 50%.

In [ ]:
array_cldpct.to_netcdf(
    os.path.join(output_dir, f'S2TS_{startdate}-{enddate}_vi-cloud_inf_{cld_max}.nc')
    )

If necessary, the following commented instructions allow you to store:
- all the VI time-series images without the application of cloud mask and filtering

In [ ]:
"""
array.to_netcdf(
    os.path.join(output_dir, f'S2TS_{startdate}-{enddate}_original.nc')
    )
"""

- all the VI time-series images with the application of cloud mask.

In [ ]:
"""
array_mask.to_netcdf(
    os.path.join(output_dir, f'S2TS_{startdate}-{enddate}_vi-mask.nc')
    )
"""